In [6]:
import pandas as pd

# wczytaj dane
df = pd.read_csv("data-complete.csv")

# użyj kolumny z możliwie dobrze oczyszczonym tekstem
df = df.dropna(subset=["text_clean"]).copy()
df["text_clean"] = df["text_clean"].astype(str).str.strip()

# opcjonalne minimum długości (żeby odsiać śmieci, np. same emotki)
df = df[df["word_count"] >= 5].reset_index(drop=True)

print(df.shape)
df[["id", "username", "text_clean"]].head()

(43219, 25)


,id,username,text_clean
0,1.0,morgenstern616,Programista Kajetan Mastela stworzył model w P...
1,2.0,bogo141,"MAZUREK: ""Nawrockiemu zginęło 77000 głosów. Gd..."
2,3.0,dorzeczy_pl,. : Karol Nawrocki przekonał wyborców Sławomir...
3,4.0,menka777,Aż w 3751 komisjach są nieprawidłowości w przy...
4,5.0,jasiu320,"W całej aferze wyborczej mi nie chodzi o to, k..."


In [7]:
!pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name)

texts = df["text_clean"].tolist()

# oblicz embeddingi
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings.shape  # (n_docs, dim)

KeyboardInterrupt: 

In [ ]:
!pip install umap-learn

  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached numba-0.65.1-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached pynndescent-0.6.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached llvmlite-0.47.0-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
Using cached umap_learn-0.5.12-py3-none-any.whl (91 kB)
Using cached numba-0.65.1-cp313-cp313-win_amd64.whl (2.8 MB)
Using cached llvmlite-0.47.0-cp313-cp313-win_amd64.whl (38.1 MB)
Using cached pynndescent-0.6.0-py3-none-any.whl (73 kB)

   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   ---------------------------------------- 0/4 [llvmlite]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [numba]
   ---------- ----------------------------- 1/4 [numba]
   ---------- -----


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import umap

umap_model = umap.UMAP(
    n_neighbors=15,
    n_components=10,   # 5–50; zacznij od 10
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

embeddings_umap = umap_model.fit_transform(embeddings)
embeddings_umap.shape  # (n_docs, 10)

d:\Code\scraping\data\data-nitter\pl-presidential-elections-2025\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(43219, 10)

In [ ]:
import numpy as np

In [ ]:
np.savez_compressed(
    "embeddings_all.npz",
    raw=embeddings,          # oryginalne MiniLM
    umap=embeddings_umap     # po UMAP
)

In [ ]:
pip install hdbscan

  Using cached hdbscan-0.8.42-cp313-cp313-win_amd64.whl.metadata (15 kB)
Using cached hdbscan-0.8.42-cp313-cp313-win_amd64.whl (670 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=50,    # 1) minimalny rozmiar klastra
    min_samples=10,         # 2) czułość na szum / gęstość
    metric='euclidean',     # 3) metryka – przy UMAP zwykle euclidean
    cluster_selection_method='eom'  # 4) metoda wyboru klastrów
)

cluster_labels = clusterer.fit_predict(embeddings_umap)
df["cluster"] = cluster_labels

NameError: name 'embeddings_umap' is not defined

In [ ]:
for mcs in [30, 50, 80, 120]:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=10,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    labels = clusterer.fit_predict(embeddings_umap)
    n_noise = (labels == -1).sum()
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"min_cluster_size={mcs} -> clusters={n_clusters}, noise={n_noise}")

NameError: name 'embeddings_umap' is not defined

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=80,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
).fit(embeddings_umap)

df["cluster"] = clusterer.labels_
df["cluster_prob"] = clusterer.probabilities_

In [ ]:
cluster_sizes = df["cluster"].value_counts().sort_values(ascending=False)
print(cluster_sizes.head(20))
noise_share = (df["cluster"] == -1).mean()
print("Udział szumu:", round(noise_share, 3))

cluster
-1     19982
 41     1847
 29     1766
 38     1669
 33     1465
 28     1360
 39     1196
 11     1015
 16      989
 34      914
 26      897
 44      880
 15      729
 10      643
 5       526
 49      429
 45      369
 31      367
 35      367
 22      343
Name: count, dtype: int64
Udział szumu: 0.462


In [ ]:
def show_cluster_examples(cluster_id, n=15, prob_thr=0.5):
    subset = df[(df["cluster"] == cluster_id) & (df["cluster_prob"] >= prob_thr)]
    print(f"Klaster {cluster_id} (rdzeń, n={len(subset)})\n")
    subset = subset.sample(n=min(n, len(subset)), random_state=42)
    for _, row in subset.iterrows():
        print(f"- @{row['username']} | {row['post_date_iso']} | interactions={row['total_interactions']:.0f}")
        print(row["text_clean"])
        print()

top_clusters = [41, 29, 38, 33, 28, 39, 11, 16, 34, 26]

for cid in top_clusters:
    show_cluster_examples(cid, n=10, prob_thr=0.5)
    input("Enter, żeby przejść do kolejnego klastra...")

Klaster 41 (rdzeń, n=1847)

- @gosia41771065 | 21-06-25 | interactions=0
💪 to jest fałszerstwo i to wystarczy aby uznać te wybory za sfałszowane

- @mach_zbigniew | 04-09-25 | interactions=3
Tak, ale pod „naszych” podciągasz wszystkich. A to jest po prostu nie ok. Sądzę, że te wybory pokazały, że nie wszyscy są „nasi”, choć za takich się uważają. Przy tak poważnych sprawach warto być precyzyjnym. Uogólnienia często szkodzą. Choć zgadzam się, że bywają nieuniknione.

- @pawel8471851111 | Jun 4, 2025 · 7:08 PM UTC | interactions=0
Nie raz, tylko łącznie jak już. Nawiążesz do tematu czy błędy będziemy wytykać?

- @henrykping42638 | 03-07-25 | interactions=0
Duda, ciągle się uczysz, a debilem zostałeś nazwany. Może nauczysz się na pamięć, że te wybory zostały sfałszowane. Rozwiać tę tezę może tylko ponowne przeliczenie wszystkich głosów. Kto nie jest za ten się tylko czegoś boi. Boisz się?

- @daniel_czyz1 | Jun 3, 2025 · 5:06 PM UTC | interactions=0
Aby nadal nic nie robić 🥳

- @janusz934

In [ ]:
import os

top_clusters = [41, 29, 38, 33, 28, 39, 11, 16, 34, 26]  # przykładowo
os.makedirs("clusters_samples_core", exist_ok=True)
N_PER_CLUSTER = 200

for cid in top_clusters:
    subset = df[(df["cluster"] == cid) & (df["cluster_prob"] >= 0.5)].copy()
    subset = subset.sample(n=min(N_PER_CLUSTER, len(subset)), random_state=42)
    subset.to_csv(f"clusters_samples_core/cluster_{cid}_core.csv", index=False)
    print("Zapisano:", cid, "->", len(subset), "tweetów")

NameError: name 'df' is not defined

In [1]:
import numpy as np
import pandas as pd

# 1. Wczytaj embeddingi
data = np.load("embeddings_all.npz")
embeddings = data["raw"]           # oryginalne embeddingi MiniLM
embeddings_umap = data["umap"]     # embeddingi po UMAP

print(embeddings.shape, embeddings_umap.shape)

# 2. Wczytaj dane tekstowe
df = pd.read_csv("data-complete.csv")

df = df.dropna(subset=["text_clean"]).copy()
df["text_clean"] = df["text_clean"].astype(str).str.strip()
df = df[df["word_count"] >= 5].reset_index(drop=True)

print(len(df))

(43219, 384) (43219, 10)
43219


In [2]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=80,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
).fit(embeddings_umap)

labels = clusterer.labels_
probs  = clusterer.probabilities_

df["cluster"] = labels
df["cluster_prob"] = probs

df["cluster"].value_counts().head(20)
(df["cluster"] == -1).mean()

np.float64(0.46234295101691386)

In [10]:
import numpy as np
import pandas as pd

# 1. Wczytaj embeddingi
data = np.load("embeddings_all.npz")
embeddings = data["raw"]           # oryginalne embeddingi MiniLM
embeddings_umap = data["umap"]     # embeddingi po UMAP

print(embeddings.shape, embeddings_umap.shape)

# 2. Wczytaj dane tekstowe
df = pd.read_csv("data-complete.csv")

df = df.dropna(subset=["text_clean"]).copy()
df["text_clean"] = df["text_clean"].astype(str).str.strip()
df = df[df["word_count"] >= 5].reset_index(drop=True)

print(len(df))

(43219, 384) (43219, 10)
43219


In [3]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=80,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
).fit(embeddings_umap)

labels = clusterer.labels_
probs  = clusterer.probabilities_

df["cluster"] = labels
df["cluster_prob"] = probs

df["cluster"].value_counts().head(20)
(df["cluster"] == -1).mean()

np.float64(0.46234295101691386)

In [5]:
df["cluster"].value_counts().sort_values(ascending=False)


cluster
-1     19982
 41     1847
 29     1766
 38     1669
 33     1465
 28     1360
 39     1196
 11     1015
 16      989
 34      914
 26      897
 44      880
 15      729
 10      643
 5       526
 49      429
 45      369
 31      367
 35      367
 22      343
 36      304
 51      277
 25      269
 6       250
 53      220
 18      218
 50      210
 30      200
 40      190
 32      187
 24      181
 43      174
 27      173
 47      169
 48      166
 2       164
 14      139
 37      136
 8       135
 21      126
 3       125
 46      125
 20      120
 0       118
 23      117
 13      109
 9       107
 42      105
 52      105
 4       102
 19       99
 7        98
 1        84
 12       83
 17       81
Name: count, dtype: int64

In [6]:
top_clusters = [41, 29, 38, 33, 28, 39, 11, 16, 34, 26]

In [7]:
import os

os.makedirs("clusters_samples_core", exist_ok=True)
N_PER_CLUSTER = 200

for cid in top_clusters:
    subset = df[(df["cluster"] == cid) & (df["cluster_prob"] >= 0.5)].copy()
    subset = subset.sample(n=min(N_PER_CLUSTER, len(subset)), random_state=42)
    subset.to_csv(f"clusters_samples_core/cluster_{cid}_core.csv", index=False)
    print("Zapisano:", cid, "->", len(subset), "tweetów")

Zapisano: 41 -> 200 tweetów
Zapisano: 29 -> 200 tweetów
Zapisano: 38 -> 200 tweetów
Zapisano: 33 -> 200 tweetów
Zapisano: 28 -> 200 tweetów
Zapisano: 39 -> 200 tweetów
Zapisano: 11 -> 200 tweetów
Zapisano: 16 -> 200 tweetów
Zapisano: 34 -> 200 tweetów
Zapisano: 26 -> 200 tweetów


In [8]:
df.to_csv("tweets_with_clusters.csv", index=False)